In [0]:
from pyspark.sql import functions as F

enriched = spark.table(
    "workspace.pyspark_deep_dive.enriched_transactions"
)

customer_metrics = (
    enriched
    .groupBy("customer_id", "customer_segment", "city")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.round(F.avg("total_amount"), 2).alias("avg_transaction_value"),
        F.countDistinct("product_id").alias("unique_products")
    )
)

customer_metrics.show(10)

In [0]:
customer_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.pyspark_deep_dive.customer_metrics"
    )

In [0]:
gold = spark.table(
    "workspace.pyspark_deep_dive.customer_metrics"
)

gold.show(5)
print("Customers:", gold.count())

In [0]:
segment_summary = (
    gold
    .groupBy("customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("customers"),
        F.round(F.sum("total_spend"), 2).alias("revenue"),
        F.round(F.avg("total_spend"), 2).alias("avg_customer_spend"),
        F.round(F.avg("avg_transaction_value"), 2).alias("avg_transaction_value")
    )
    .orderBy(F.col("revenue").desc())
)

segment_summary.show()

In [0]:
city_summary = (
    gold
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("customers"),
        F.round(F.sum("total_spend"), 2).alias("revenue"),
        F.round(F.avg("total_spend"), 2).alias("avg_customer_spend")
    )
    .orderBy(F.col("revenue").desc())
)

city_summary.show()

In [0]:
segment_data = [
    row.asDict()
    for row in segment_summary.collect()
]

city_data = [
    row.asDict()
    for row in city_summary.collect()
]

print(segment_data)
print(city_data)

In [0]:
import json

prompt = f"""
You are a financial data analyst.

Analyze the following customer transaction summaries.

Customer Segment Summary:
{json.dumps(segment_data, default=str)}

City Summary:
{json.dumps(city_data, default=str)}

Provide:
1. Top business insight
2. Highest revenue customer segment
3. Highest revenue city
4. One actionable observation

Keep the response concise and data-driven.
"""

print(prompt)

In [0]:
insight_input = {
    "segment_summary": segment_data,
    "city_summary": city_data
}

print(json.dumps(insight_input, indent=2, default=str))

In [0]:
def generate_business_prompt(data):
    return f"""
You are a financial data analyst.

Analyze this transaction data:

{json.dumps(data, indent=2, default=str)}

Return:
- Key revenue insight
- Customer segment insight
- Geographic insight
- One actionable recommendation

Use only the supplied data.
Do not invent numbers.
Keep it under 150 words.
"""

ai_prompt = generate_business_prompt(insight_input)

print(ai_prompt)

In [0]:
print(ai_prompt)

In [0]:
%sql
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Explain why customer segmentation is useful in financial transaction analytics. Give 2 concise points.'
) AS ai_response;

In [0]:
%sql
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'YOUR BUSINESS ANALYSIS PROMPT'
) AS ai_response;

In [0]:
import json

business_data = json.dumps(
    {
        "customer_segments": segment_data,
        "cities": city_data
    },
    default=str
)

final_prompt = f"""
You are a financial data analyst.

Analyze the following transaction analytics generated using PySpark.

DATA:
{business_data}

Provide:
1. Key revenue insight
2. Customer segment insight
3. Geographic insight
4. One actionable observation

Rules:
- Use only the supplied data.
- Do not invent numbers.
- Mention relevant numbers when useful.
- Keep the answer under 150 words.
"""

print(final_prompt)

In [0]:
final_prompt = f"""
You are a financial data analyst.

Analyze the following PySpark-generated transaction analytics.

DATA:
{business_data}

Provide exactly these sections:

Key Revenue Insight:
Customer Segment Insight:
Geographic Insight:
Actionable Observation:

Rules:
- Use only the supplied data.
- Do not invent numbers.
- Currency is USD.
- Use million/billion notation where appropriate.
- Keep the answer under 150 words.
"""

In [0]:
response = spark.sql("""
SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    :prompt
) AS business_insights
""", args={"prompt": final_prompt})

response.show(truncate=False)